# Sarcasm Detection in Reddit Comments

Этот проект сфокусирован на обнаружении сарказма в комментариях на Реддите, используя классические техники NLP.

Цель проекта заключается в построении бейзлайна, анализе его поведения и улучшении результатов.

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.base import clone


## Data Loading

Мы загружаем датасет с комментариями с Реддита и исследуем его структуру.

In [2]:
df = pd.read_csv('../Sarcasm_project/train-balanced-sarcasm.csv')
df.head()

,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment
0,0,NC and NH.,Trumpbart,politics,2,-1,-1,2016-10,2016-10-16 23:55:23,"Yeah, I get that argument. At this point, I'd ..."
1,0,You do know west teams play against west teams...,Shbshb906,nba,-4,-1,-1,2016-11,2016-11-01 00:24:10,The blazers and Mavericks (The wests 5 and 6 s...
2,0,"They were underdogs earlier today, but since G...",Creepeth,nfl,3,3,0,2016-09,2016-09-22 21:45:37,They're favored to win.
3,0,"This meme isn't funny none of the ""new york ni...",icebrotha,BlackPeopleTwitter,-8,-1,-1,2016-10,2016-10-18 21:03:47,deadass don't kill my buzz
4,0,I could use one of those tools.,cush2push,MaddenUltimateTeam,6,-1,-1,2016-12,2016-12-30 17:00:13,Yep can confirm I saw the tool they use for th...


In [3]:
df.columns

Index(['label', 'comment', 'author', 'subreddit', 'score', 'ups', 'downs',
       'date', 'created_utc', 'parent_comment'],
      dtype='object')

## Exploratory Data Analysis

Мы исследуем датасет по:

- количеству образцов

- балансу классов

- распределению длин комментариев

In [4]:
print(f"""Датасет сбалансированный:\n {df['label'].value_counts()[0], df['label'].value_counts()[1]}.
\n общий размер {df.shape[0]} по {df.shape[1]} колонок.
\n Самый длинный комментарий длинной {df['comment'].str.len().max()} символов 
\n Самый короткий — {df['comment'].str.len().min()} символов """)

Датасет сбалансированный:
 (505413, 505413).

 общий размер 1010826 по 10 колонок.

 Самый длинный комментарий длинной 10000.0 символов 

 Самый короткий — 1.0 символов 


In [5]:
print("Sarcastic examples:\n")
print(df[df['label'] == 1]['comment'].sample(3, random_state=17).values)

print("\nNot sarcastic examples:\n")
print(df[df['label'] == 0]['comment'].sample(3, random_state=17).values)

Sarcastic examples:

["prolly just forgot some 0's on the end" 'Thanks Renzie!'
 'She was a moderate up until 10/13/2015']

Not sarcastic examples:

['not enough asians in the picture....' 'It works'
 'The novel Frankenstein, by inspiring the later films, made most of the population believe the monster to be named Frankenstein and not the scientist.']


## Data Preprocessing

Мы чистим текст путем:

- Привидения текста к нижнему регистру

- Убирания специальных символов

- Убирания излишних пробелов

*Так же из-за большого размера датасета (~1 миллион признаков),  был использован случайный сабсет размером 100 000 признаков для быстрого эксперимента.*

In [6]:
df = df.dropna(subset=['comment', 'label']).copy()

df_sample = df.sample(
    n=100000,
    random_state=17,
).copy()

In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [8]:
df_sample['clean_comment'] = df_sample['comment'].apply(clean_text)

In [9]:
df_sample[['clean_comment', 'comment']].head()

,clean_comment,comment
469600,starting to feel pretty fucking tired of all t...,Starting to feel pretty fucking tired of all t...
639138,the room,The Room.
240294,my bf has inches long and its pretty thick,My BF has 10 inches long and its pretty thick.
920561,correlation does not equal causation,CORRELATION DOES NOT EQUAL CAUSATION
944985,i had a similar type of experience though i ma...,"I had a similar type of experience, though I m..."


In [10]:
df_sample[(df_sample['clean_comment'] == '')]

,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment,clean_comment
408073,1,9/11.,PopeyeSamurai,GlobalOffensive,-2,-2,0,2016-01,2016-01-14 22:55:41,I really doubt that. ESPN is American and you ...,
910106,0,:-(,TaylorS1986,worldnews,11,11,0,2013-04,2013-04-17 01:30:52,Consulate: Chinese national is 3rd victim in B...,
393621,0,*2016-01-01,ThePiachu,pics,-1,-1,0,2016-01,2016-01-01 20:59:12,First time. Fucking nailed it.,
474845,0,"726,042",kqxrl,counting,3,3,0,2016-01,2016-01-18 16:33:27,726 041,
929079,0,+1,skellener,iphone,2,2,0,2013-04,2013-04-19 03:43:33,It's the best iPhone reddit browser there is I...,
...,...,...,...,...,...,...,...,...,...,...,...
459391,0,1/51,EyyBBWntSumFuk,OkCupid,1,1,0,2016-02,2016-02-23 15:33:07,Black and brown dudes on okcupid...what's your...,
355842,0,@@@@@@@@@@@@@@@@,PrideSax711,leagueoflegends,0,0,0,2016-01,2016-01-20 14:51:00,Does it bother anyone else that Mafia Graves u...,
131838,0,;),Creepersteak,malefashionadvice,3,-1,-1,2016-12,2016-12-16 19:29:02,a lot of things are 2 fun 4 my face,
311587,1,....,9000cody,xboxone,4,4,0,2016-03,2016-03-01 00:38:18,Glad they're making a sequel to the most loved...,


In [11]:
def count_empty_strings(text: pd.Series) -> None:
    empty_count = text.fillna('').str.strip().eq('').sum()
    total = len(text)

    print(f'Пустых строк: {empty_count}')
    print(f'Доля: {empty_count / total:.4f}')

In [12]:
count_empty_strings(df_sample['clean_comment'])

Пустых строк: 245
Доля: 0.0024


In [13]:
df_sample = df_sample[df_sample['clean_comment'].str.strip() != '']

In [14]:
df_sample = df_sample.reset_index(drop=True)

In [15]:
count_empty_strings(df_sample['clean_comment'])

Пустых строк: 0
Доля: 0.0000


In [16]:
df_sample[(df_sample['clean_comment'] == '')]

,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment,clean_comment


In [17]:
duplicate_mask = df_sample.duplicated(
    subset='clean_comment',
    keep=False,
)

print(f'строк дубликатов: {duplicate_mask.sum()}')
print(f'Доля дубликатов: {duplicate_mask.mean():.2%}')

df_sample = (
    df_sample.loc[~duplicate_mask].reset_index(drop=True)
)

строк дубликатов: 5231
Доля дубликатов: 5.24%


## Train/Validation/Test Split

Разделяем датасет на тренировачный и тестовый.

In [18]:
X = df_sample['clean_comment']
y = df_sample['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=17,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=17,
    stratify=y_temp,
)

In [19]:
df_sample['split'] = ''

df_sample.loc[X_train.index, 'split'] = 'train'
df_sample.loc[X_val.index, 'split'] = 'validation'
df_sample.loc[X_test.index, 'split'] = 'test'

print(df_sample['split'].value_counts(), '\n')
print(pd.crosstab(df_sample['split'], df_sample['label'], normalize='index'))

assert (df_sample['split'] != '').all()

df_sample.to_csv("./df_sample.csv", index=False)

split
train         66166
test          14179
validation    14179
Name: count, dtype: int64 

label              0         1
split                         
test        0.492207  0.507793
train       0.492232  0.507768
validation  0.492207  0.507793


In [20]:
experiment_results = []

dummy = DummyClassifier(
    strategy='stratified',
    random_state=17,
)

dummy.fit(
    np.zeros((len(y_train), 1)),
    y_train,
)

dummy_pred = dummy.predict(
    np.zeros((len(y_val), 1))
)

dummy_proba = dummy.predict_proba(
    np.zeros((len(y_val), 1))
)[:,1]

dummy_metrics = {
        'model': 'dummy',
        'accuracy': accuracy_score(y_val, dummy_pred),
        'f1': f1_score(y_val, dummy_pred),
        'roc_auc': roc_auc_score(y_val, dummy_proba),
    }

experiment_results.append(dummy_metrics)

In [21]:
print(experiment_results)

[{'model': 'dummy', 'accuracy': 0.4975668241765992, 'f1': 0.5084184377587634, 'roc_auc': 0.49734357835411}]


In [22]:
def evaluate_model(name, vectorizer_params, model_params=None):
    model_params = model_params or {}

    pipeline = Pipeline([
        (
            'tfidf',
            TfidfVectorizer(**vectorizer_params),
        ),
        (
            'classifier',
            LogisticRegression(**model_params),
         )
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)
    y_proba = pipeline.predict_proba(X_val)[:,1]

    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_val, y_pred),
        'f1': f1_score(y_val, y_pred),
        'roc_auc': roc_auc_score(y_val, y_proba),
    }

    return metrics, pipeline

## Baseline Model

Используем TF-IDF векторизацию and Логистическую Регресию как бейзлайн.

In [23]:
experiment_baseline_params={
    'name': 'baseline',
    'vectorizer_params': {
        'max_features': 10000,
    },
    'model_params': {
        'max_iter': 1000,
    },
}

experiment_baseline, experiment_baseline_model = evaluate_model(**experiment_baseline_params)

experiment_results.append(experiment_baseline)

pd.DataFrame([experiment_baseline])

,model,accuracy,f1,roc_auc
0,baseline,0.66937,0.666049,0.728043


## Эксперимент 1: Добавление n-грамм

Мы расширяем TF-IDF биграммами для выявления закономерностей на уровне фраз.

In [24]:
experiment_01_params = {
    'name':'exp_01_more_ngram',
    'vectorizer_params':{
        'max_features': 10000,
        'ngram_range': (1,2),
        'min_df': 5,
        },
    'model_params':{
        'max_iter': 1000,
        }    
    }
experiment_01, experiment_01_model = evaluate_model(**experiment_01_params)

experiment_results.append(experiment_01)

pd.DataFrame([experiment_01])


,model,accuracy,f1,roc_auc
0,exp_01_more_ngram,0.68143,0.676456,0.744247


In [25]:
print(*experiment_results, sep='\n')

{'model': 'dummy', 'accuracy': 0.4975668241765992, 'f1': 0.5084184377587634, 'roc_auc': 0.49734357835411}
{'model': 'baseline', 'accuracy': 0.6693701953593342, 'f1': 0.666049294771335, 'roc_auc': 0.7280430378436897}
{'model': 'exp_01_more_ngram', 'accuracy': 0.6814302842231469, 'f1': 0.6764558412721152, 'roc_auc': 0.7442472855073156}


## Эксперимент 2: Увеличение max_features

Мы увеличиваем количество признаков, чтобы обеспечить представление как униграмм, так и биграмм.

In [26]:
experiment_02_params = {
    'name':'exp_02_more_features',
    'vectorizer_params':{
        'max_features': 20000,
        'ngram_range': (1,2),
        'min_df': 5,
        },
    'model_params':{
        'max_iter': 1000,
        }    
    }
experiment_02, experiment_02_model = evaluate_model(**experiment_02_params)

experiment_results.append(experiment_02)

pd.DataFrame([experiment_02])

,model,accuracy,f1,roc_auc
0,exp_02_more_features,0.68411,0.680277,0.7472


In [27]:
experiment_results_df = pd.DataFrame(experiment_results)

experiment_results_df = (
    experiment_results_df
    .sort_values(by='f1', ascending=False)
    .reset_index(drop=True)
)

experiment_results_df

,model,accuracy,f1,roc_auc
0,exp_02_more_features,0.684110,0.680277,0.747200
1,exp_01_more_ngram,0.681430,0.676456,0.744247
2,baseline,0.669370,0.666049,0.728043
3,dummy,0.497567,0.508418,0.497344


In [28]:
experiment_models = {
    experiment_baseline['model']: experiment_baseline_model,
    experiment_01['model']: experiment_01_model,
    experiment_02['model']: experiment_02_model,
}

best_model_name = experiment_results_df.loc[0, 'model']
best_validation_model = experiment_models[best_model_name]

print('Лучшая модель:', best_model_name)

Лучшая модель: exp_02_more_features


In [29]:
best_val_pred = best_validation_model.predict(X_val)
best_val_proba = best_validation_model.predict_proba(X_val)[:, 1]

print('Best validation model:', best_model_name, '\n')
print(classification_report(y_val, best_val_pred))

Best validation model: exp_02_more_features 

              precision    recall  f1-score   support

           0       0.67      0.71      0.69      6979
           1       0.70      0.66      0.68      7200

    accuracy                           0.68     14179
   macro avg       0.68      0.68      0.68     14179
weighted avg       0.68      0.68      0.68     14179



## Анализ Ошибок Модели

Анализируем ошибки модели, чтобы понять её ограничения.

In [30]:
results = pd.DataFrame({
    'text': X_val.reset_index(drop=True),
    'true': y_val.reset_index(drop=True),
    'pred': best_val_pred
})

mistakes = results[results['true'] != results['pred']]
mistakes.head(20)

,text,true,pred
0,im aware but that still only damage and healin...,0,1
4,lightning,1,0
8,i hope someone buys it and turns it into a condo,1,0
19,oh clever dota takes more skills comment witho...,1,0
20,why they dont even play similar,1,0
26,its an atheist thing you wouldnt get it,1,0
27,oh no not in my backyard,1,0
36,the i,1,0
37,but you can get married there,1,0
39,the way some people talk i think that laffer c...,1,0


In [31]:
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

final_model = clone(best_validation_model)
final_model.fit(X_train_val, y_train_val)

y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)[:, 1]

print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print("Test F1:", f1_score(y_test, y_test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_proba))
print(classification_report(y_test, y_test_pred))

Test accuracy: 0.6887650751110798
Test F1: 0.6841766263508194
Test ROC-AUC: 0.7534895460190094
              precision    recall  f1-score   support

           0       0.67      0.71      0.69      6979
           1       0.71      0.66      0.68      7200

    accuracy                           0.69     14179
   macro avg       0.69      0.69      0.69     14179
weighted avg       0.69      0.69      0.69     14179



## Результаты

| Model                     | F1-score |
|--------------------------|---------|
| Baseline                 | 0.6687  |
| + n-граммы               | 0.6805  |
| + больше признаков       | 0.6833  |

## Заключение

- TF-IDF обеспечивает надежную базовую модель
- Увеличение пространства признаков улучшило производительность
- Модель испытывает трудности с контекстно-зависимым сарказмом

Дальнейшая работа:
- Использование родительских комментариев
- Использование моделей на основе трансформеров (например, BERT)